In [1]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

import faiss
from langchain.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from pythainlp.tokenize import word_tokenize

import pandas as pd
import numpy as np

from tqdm import tqdm
import torch
import json

ROOT_DIR = "/project/lt200304-dipmt/paweekorn"
MODEL_ID = "qwen3-4b"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 09-22 21:03:17 [__init__.py:241] Automatically detected platform cuda.


## Overview

In [2]:
train_df = pd.read_csv(f"{ROOT_DIR}/data/train_40k.csv")
test_df = pd.read_csv(f"{ROOT_DIR}/data/test_set.csv")

with open(f"{ROOT_DIR}/data/wipo/WIPO.json", "r") as f:
    wipo_data = json.load(f)
    wipo_data = {int(k): v for k, v in wipo_data.items()}

train_df['WIPO'] = train_df['NAME'].map(wipo_data)
test_df['WIPO'] = test_df['NAME'].map(wipo_data)

print(train_df.shape)
train_df.head()

(39706, 4)


,ENG,THA,NAME,WIPO
0,"condiment, namely, pepper sauce","เครื่องปรุงรส, คือ, ซอสพริกไทย",30,"Coffee, tea, cocoa and substitutes therefor; r..."
1,machine and motor oil,น้ำมันเครื่องและมอเตอร์,4,Industrial oils and greases; lubricants; dust ...
2,medical analysis for diagnostic or treatment p...,วิเคราะห์ทางการแพทย์เพื่อการวินิจฉัยหรือการรักษา,44,Medical services; veterinary services; hygieni...
3,cleaning preparation for window pane,สารเตรียมขึ้นสำหรับทำความสะอาดบานกระจกหน้าต่าง,3,Non-medicated cosmetics and toiletry preparati...
4,protective padding for sport,แผ่นป้องกันสำหรับกีฬา,28,"Games, toys and playthings; video game apparat..."


In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"{ROOT_DIR}/models/base/{MODEL_ID}",
    max_seq_length = 2048,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = True,
    device_map="auto",
)

==((====))==  Unsloth 2025.8.4: Fast Qwen3 patching. Transformers: 4.55.0. vLLM: 0.10.1.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.496 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Data Prep

In [ ]:
with open(f"{ROOT_DIR}/data/prompt/base_prompt.txt") as f:
    instruction = f.read()

def get_relevant_docs(query, k=4):
    query_embedding = embeddings.embed_query(query)
    docs = vectorstore.similarity_search_by_vector(query_embedding, k=k)
    
    relevant = "\n**Retrieved References:**\n"
    for i, doc in enumerate(docs[1:]):
        relevant += f'''
English: {doc.page_content}
Thai: {doc.metadata['thai']}
'''
    return relevant

def formatting_prompt(df):
    batch = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        prompt = [
            { "role": "user", "content": instruction.format(
                WIPO=row['WIPO'], 
                RAG_DOC=get_relevant_docs(row['ENG'], k=3), 
                ENGLISH=row['ENG']) }, 
            { "role": "assistant", "content": 
              f'''{{"thai_translation": "{row['THA']}" }}''' 
            }
        ]
        message = tokenizer.apply_chat_template(
            prompt, tokenize=False,
            add_generation_prompt=False,
        )
        batch.append({'text': message})

    return Dataset.from_list(batch)

train_set = formatting_prompt(train_df)
test_set = formatting_prompt(test_df)
print(train_set['text'][0])

## Model Training

In [6]:
# model = FastLanguageModel.get_peft_model(
#     model, 
#     r = 64,           
#     lora_alpha = 64,  
#     lora_dropout = 0,
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     bias = "none",
#     random_state = 3407,
# )

In [8]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=f"{ROOT_DIR}/models/fine-tuned/{MODEL_ID}",
    logging_dir=f"{ROOT_DIR}/models/fine-tuned/{MODEL_ID}/logs",
    dataset_text_field = "text",
    per_device_train_batch_size = 32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps = 2,
    warmup_ratio = 0.03,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    logging_steps = 100,
    eval_steps = 100,
    eval_strategy = "steps",
    save_steps = 210,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_set,
    eval_dataset = test_set,
    args = args,
    gradient_checkpoint=True,
)

train_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/39706 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/8392 [00:00<?, ? examples/s]

[2025-09-22 21:05:15,882] [INFO] [real_accelerator.py:260:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/lustrefs/disk/home/psoratya/.conda/envs/unsloth_env/bin/../lib/gcc/x86_64-conda-linux-gnu/12.4.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/lustrefs/disk/home/psoratya/.conda/envs/unsloth_env/bin/../lib/gcc/x86_64-conda-linux-gnu/12.4.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-09-22 21:05:22,627] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 39,706 | Num Epochs = 1 | Total steps = 621
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 4,022,468,096 of 4,022,468,096 (100.00% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.476800,0.304354
200,0.253900,0.269237
300,0.228100,0.249585
400,0.204200,0.232950
500,0.197800,0.225659
600,0.196100,0.225445


Unsloth: Not an error, but Qwen3Model does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


## Merged model

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

root_dir = "/project/lt200304-dipmt/paweekorn/models"
model_id = "qwen3-8b"

base_model = AutoModelForCausalLM.from_pretrained(
    f"{root_dir}/base/{model_id}",
    device_map="auto",
    trust_remote_code=False
)
tokenizer = AutoTokenizer.from_pretrained(f"{root_dir}/base/{model_id}")

merged_model = PeftModel.from_pretrained(base_model, f"{root_dir}/adapter/{model_id}")
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(f"{root_dir}/fine-tuned/{model_id}", safe_serialization=True)
tokenizer.save_pretrained(f"{root_dir}/fine-tuned/{model_id}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

('/project/lt200304-dipmt/paweekorn/models/fine-tuned/eirmed-8b/tokenizer_config.json',
 '/project/lt200304-dipmt/paweekorn/models/fine-tuned/eirmed-8b/special_tokens_map.json',
 '/project/lt200304-dipmt/paweekorn/models/fine-tuned/eirmed-8b/chat_template.jinja',
 '/project/lt200304-dipmt/paweekorn/models/fine-tuned/eirmed-8b/tokenizer.json')